# QUESTION 3 Total Marks [18]

Dummy + Logistic Regression + **Random Forest** + **MLP** + **tabular embedding net** on the frozen Drive split (seed 42). Baseline does **not** count as one of the four.

Locks: `prep.py` grouped features; main split stratified 70/15/15; no test look; no champion; no Streamlit.

Report: `/cursor/stores/self/docs/q3.md`. Run from the **repo root**.


In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

for root in (Path("/workspace"), Path(".."), Path(".")):
    if (root / "src" / "prep.py").exists() and (root / "data" / "processed" / "train.parquet").exists():
        os.chdir(root)
        sys.path.insert(0, str(root.resolve()))
        break

print("cwd", Path.cwd())
from src.prep import FEATURES, SEED
print("SEED", SEED, "FEATURES", FEATURES)


## 3.1 Build a transparent naïve baseline (for example, majority/stratified DummyClassifier) and explain what it establishes. (2)


In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score

train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/val.parquet")
y_tr = train["target"].astype(str)
y_va = val["target"].astype(str)
CLASSES = ["Resolved Fast", "Resolved Slow", "Unresolved"]

rows = []
for strategy in ("most_frequent", "stratified"):
    d = DummyClassifier(strategy=strategy, random_state=SEED)
    d.fit(np.zeros((len(train), 1)), y_tr)
    pred = d.predict(np.zeros((len(val), 1)))
    print("\n", strategy)
    print(classification_report(y_va, pred, labels=CLASSES, zero_division=0))
    rows.append({"model": f"Dummy ({strategy})", "macro_f1_val": f1_score(y_va, pred, average="macro")})
pd.DataFrame(rows)


Dummy **ignores features**. Most-frequent val macro-F1 ≈ 0.268 (accuracy 0.674, Slow/Unresolved recall 0). Stratified ≈ 0.333. That is the floor. Dummy is **not** one of the four models. Test is not scored here.


## 3.2 Train at least four substantive models on identical data splits: at least two traditional models and at least two neural-learning models. The naïve baseline does not count among the four. (10)

LR on **full train**. Forest / MLP / embedding on a documented **150,000-row train subsample**. Official val stays full. Test is not loaded.

If `experiments/q3_val_metrics.csv` already exists (this VM), the next cell loads it. Delete that file and re-run to train from scratch (`python -m src.train_q3`).


In [ ]:
from pathlib import Path
p = Path("experiments/q3_val_metrics.csv")
if p.exists():
    val_df = pd.read_csv(p)
    print("loaded", p)
else:
    from src.train_q3 import main
    val_df = main()
val_df


Four substantive models: Logistic Regression, Random Forest, MLP, EmbeddingNet. Figure: `/cursor/stores/self/media/q3/fig_q3_val_macro_f1.png` (also `figures/fig_q3_val_macro_f1.png`). This table is **not** a champion pick.


## 3.3 Traditional examples include Logistic Regression, Decision Tree/Random Forest, Support Vector Machine, Naïve Bayes or gradient boosting. Neural requirements include an MLP and one task-appropriate second architecture (for example, CNN for imagery, GRU/LSTM for sequences, or an approved tabular neural network). (3)

- Traditional: **Logistic Regression** + **Random Forest** (locked; not boosting).
- Neural: **MLP** (required) + **tabular embedding net** (lecturer-approved). Not GRU/LSTM (no sequences). Not CNN (no images).


## 3.4 Document architecture/hyperparameters, loss, optimiser, epochs, batch size, regularisation, early stopping, seeds, compute environment and reproducible training procedure. (5)


In [ ]:
import json
print(Path("experiments/q3_compute.json").read_text())
print("MLP history")
print(pd.read_csv("experiments/mlp_history.csv").to_string(index=False))
print("Embedding history")
print(pd.read_csv("experiments/embedding_history.csv").to_string(index=False))
print("hyperparameters are logged in experiments/q3_experiment_log.csv")
pd.read_csv("experiments/q3_experiment_log.csv")[["run_id","model","seed","train_rows_used","runtime_sec_fit","notes"]]


Shared: seed 42, `prep.py` on train only, 1,083 sparse columns. LR: saga, C=1, balanced, max_iter=80, full train. RF: 40 trees, depth 12, leaf 20, n=150,000. MLP: 1083→128→64→3, dropout 0.2, Adam 1e-3, weight decay 1e-4, CE class weights, batch 2048, max 8 epochs, patience 2 on **15% of the train subsample**. Embedding: emb_dim=12, hidden 64, batch 4096, same stopping. Compute: CPU PyTorch, ~16 GB RAM. Histories: `/cursor/stores/self/media/q3/fig_q3_mlp_history.png`, `fig_q3_embedding_history.png`.


## 3.5 Provide a concise rationale linking each model family to the data and problem. (2)

- **Dummy** — floor from the 67% Fast mix.
- **Logistic Regression** — linear, sparse, readable type/department effects; full train fits.
- **Random Forest** — type × department × weekend interactions; Forest is the locked traditional-2.
- **MLP** — required non-linear net on the **same** one-hot table.
- **Embedding net** — high-cardinality suburb_group / ward / complaint type; lecturer-approved second neural architecture.

Q4 compares fairly. Q5 tunes only the val winner. Test stays locked.
